In [2]:
import hashlib, struct, zipfile, requests, pandas as pd
import math, os, time, itertools, string, hmac, base64
import pyotp, qrcode
from IPython.display import display
from cryptography.hazmat.primitives.asymmetric import ed25519, padding, rsa, utils
from cryptography.hazmat.primitives import serialization, hashes
from cryptography import x509

# Global helper for consistency
def SHA1(text): 
    return hashlib.sha1(text.encode()).hexdigest().upper()


# Workshop: Authentication and Keys 🔐

---
## 0. Password Database Leak

**Question:** If these databases leaked, what could an attacker find out?


In [3]:
# Leaked databases for comparison
leak_a = pd.DataFrame([
    {"source": "Local Forum", "user": "alice", "password": "Spring2024!"},
    {"source": "Local Forum", "user": "bob", "password": "123456"},
    {"source": "Local Forum", "user": "carol", "password": "password"},
    {"source": "Local Forum", "user": "dave", "password": "dragon"},
])

leak_b = pd.DataFrame([
    {"source": "Global Shop", "email": "alice@example.com", "password": "Spring2024!"},
    {"source": "Global Shop", "email": "bob@provider.net", "password": "letmein123"},
    {"source": "Global Shop", "email": "eve@security.org", "password": "Complex_123_Secret"},
])

leak_c = pd.DataFrame([
    {"source": "alice", "hash": "7110EDA4D09E062AA5E4A390B0A572AC0D2C0220"},
    {"user": "carol", "hash": "5BAA61E4C9B93F3F0682250B6CF8331B7EE68FD8"},
    {"user": "eve", "hash": "4C68B0A58442F9943B8808DED773D4A889417088"},
])

display(leak_a)
display(leak_b)
display(leak_c)

,source,user,password
0,Local Forum,alice,Spring2024!
1,Local Forum,bob,123456
2,Local Forum,carol,password
3,Local Forum,dave,dragon


,source,email,password
0,Global Shop,alice@example.com,Spring2024!
1,Global Shop,bob@provider.net,letmein123
2,Global Shop,eve@security.org,Complex_123_Secret


,source,hash,user
0,alice,7110EDA4D09E062AA5E4A390B0A572AC0D2C0220,NaN
1,NaN,5BAA61E4C9B93F3F0682250B6CF8331B7EE68FD8,carol
2,NaN,4C68B0A58442F9943B8808DED773D4A889417088,eve


---
## 1. Am I Compromised?

### Task 1: Browser check
Open your browser and check your email at:
### 👉 [haveibeenpwned.com](https://haveibeenpwned.com)

---
### Task 2: How to ask about a secret without revealing it?
We use **k-Anonymity**. You only send the first 5 characters of your password's SHA-1 hash.

**The Math of SHA-1:**
- SHA-1 produces a **160-bit** digest.
- In Hexadecimal, this is **40 characters** (each hex char = 4 bits).
- We send 5 hex chars ($16^5 = 1,048,576$ possibilities).


In [4]:
def check_password_safe(password):
    # Calculate hash
    digest_bytes = hashlib.sha1(password.encode()).digest()
    sha1 = digest_bytes.hex().upper()
    
    prefix, suffix = sha1[:5], sha1[5:]
    
    print(f"Password:   '{password}'")
    print(f"SHA-1 Hash:  {sha1}")
    print(f"Size:        {len(digest_bytes)} bytes / {len(digest_bytes)*8} bits")
    print("-" * 40)
    print(f"Sent to API: {prefix}")

    res = requests.get(f"https://api.pwnedpasswords.com/range/{prefix}")
    hashes = {line.split(':')[0]: int(line.split(':')[1]) for line in res.text.splitlines()}
    
    count = hashes.get(suffix, 0)
    print(f"\n⚠️  Pwned {count:,} times!" if count else "\n✅ Not found.")

check_password_safe("password123")

Password:   'password123'
SHA-1 Hash:  CBFDAC6008F9CAB4083784CBD1874F76618D2A97
Size:        20 bytes / 160 bits
----------------------------------------
Sent to API: CBFDA

⚠️  Pwned 2,254,650 times!


---
## 2. Properties of Hash Functions

**One-wayness:** You can't reverse the math.
**Avalanche Effect:** A tiny change in input creates a completely different output.

**Task:** Compare two similar passwords.

In [5]:
def SHA1(text): return hashlib.sha1(text.encode()).hexdigest().upper()

p1, p2 = "password123", "password124"
print(f"'{p1}' -> {SHA1(p1)}")
print(f"'{p2}' -> {SHA1(p2)}")

'password123' -> CBFDAC6008F9CAB4083784CBD1874F76618D2A97
'password124' -> 2B4BFCC447C3C8726D26C22927A68F511D5E01CC


---
## 3. Practical Attack: The Zip File 🔓

We have a locked file: `files/file.zip`.

### 3.1 The "Online" Attack (Slow)
We try to open the zip file by guessing. This is slow because the program must interact with the file system for every guess.


In [6]:
ZIP = "files/file.zip"
charset = string.ascii_lowercase
start_time = time.time()
attempts_online = 0

print("Attacking Zip via extraction (Online/System)...")

for combo in itertools.product(charset, repeat=4):
    attempts_online += 1
    guess = "".join(combo)
    try:
        with zipfile.ZipFile(ZIP) as zf:
            zf.read(zf.namelist()[0], pwd=guess.encode())
            print(f"🔓 SUCCESS! Password is: {guess}")
            break
    except:
        continue

elapsed_online = time.time() - start_time
speed_online = attempts_online / elapsed_online if elapsed_online > 0 else 0

print(f"Attempts: {attempts_online}")
print(f"Time:     {elapsed_online:.4f} seconds")
print(f"Speed:    {speed_online:,.2f} guesses/s")

Attacking Zip via extraction (Online/System)...
🔓 SUCCESS! Password is: abcd
Attempts: 732
Time:     0.1038 seconds
Speed:    7,048.67 guesses/s


---
### 3.2 The "Offline" Attack (Fast)

In a real leak, attackers extract the internal "fingerprint" (the hash) and attack it in memory.
Let's see the **Check Byte** from the Zip header, then simulate a full hash attack.


In [ ]:
# 1. Extraction from file header
with open(ZIP, "rb") as f:
    raw = f.read()
fname_len, extra_len = struct.unpack_from("<HH", raw, 26)
check_byte = raw[30 + fname_len + extra_len + 11]

print(f"Extracted Check Byte: 0x{check_byte:02x}")
print("This byte allows the system to verify the password without decrypting the whole file.")

# 2. Mathematical Attack (Simulation using SHA-1)
target_hash = SHA1("abcd") 
start_time = time.time()
attempts_offline = 0

print("\nStarting mathematical attack on the hash...")

for combo in itertools.product(charset, repeat=4):
    attempts_offline += 1
    if SHA1("".join(combo)) == target_hash:
        print(f"💥 CRACKED! Password is: {''.join(combo)}")
        break

elapsed_offline = time.time() - start_time
speed_offline = attempts_offline / elapsed_offline if elapsed_offline > 0 else 0

# 3. Comparison Table
data = []
if 'speed_online' in globals() and speed_online > 0:
    data.append({"Method": "Online (Zip)", "Speed (guesses/s)": speed_online, "Time": elapsed_online, "Attempts": attempts_online})
data.append({"Method": "Offline (Hash)", "Speed (guesses/s)": speed_offline, "Time": elapsed_offline, "Attempts": attempts_offline})

display(pd.DataFrame(data))

if 'speed_online' in globals() and speed_online > 0:
    print(f"\n🚀 The Offline attack is {speed_offline / speed_online:.1f}x faster!")

---
## 4. Measuring Your Strength
How long would it take to crack *your* password using the "Offline" speed we just measured?


In [ ]:
def estimate_cracking(pw, speed):
    charset_size = 0
    if any(c.islower() for c in pw): charset_size += 26
    if any(c.isupper() for c in pw): charset_size += 26
    if any(c.isdigit() for c in pw): charset_size += 10
    if any(not c.isalnum() for c in pw): charset_size += 32
    
    combinations = charset_size ** len(pw)
    seconds = combinations / speed if speed > 0 else 0
    
    print(f"Password: {pw}")
    print(f"Combinations: {combinations:,.0f}")
    
    if seconds == 0:
        print("Run the calibration in Section 3.2 first!")
    elif seconds < 60:
        print(f"Estimated cracking time: {seconds:.2f} seconds")
    elif seconds < 86400:
        print(f"Estimated cracking time: {seconds/3600:.2f} hours")
    else:
        print(f"Estimated cracking time: {seconds/86400:,.0f} days")

user_pw = input("Enter a password to test: ")
# Use speed_offline from section 3.2
if 'speed_offline' in globals():
    estimate_cracking(user_pw, speed_offline)
else:
    print("Error: Please run Section 3.2 first to calibrate speed.")

---
## 5. Dynamic Secrets: TOTP (Mobile Check) 📱

### Task 1: Phone Sync

In [ ]:
secret = pyotp.random_base32()
totp = pyotp.TOTP(secret)

print("1. Open Google Authenticator (or similar app).")
print("2. Tap '+' and choose 'Scan a QR code'.")
display(qrcode.make(totp.provisioning_uri(name="student@muni.cz", issuer_name="PoznejFI")))

user_code_raw = input("Enter the 6-digit code from your phone: ")
user_code = ''.join(filter(str.isdigit, user_code_raw)).strip()[:6]

if totp.verify(user_code):
    print("✅ Verification Success!\n")
    # Show secret key details
    secret_bytes = base64.b32decode(secret, casefold=True)
    print(f"Key (Base32): {secret}")
    print(f"Key (Hex):    {secret_bytes.hex()}")
    print(f"Key (Bits):   {''.join(f'{b:08b}' for b in secret_bytes)}")
    
    # Calculate strength
    key_combinations = 32 ** len(secret)
    print(f"Combinations: {key_combinations:,.0f}")
    if 'speed_offline' in globals():
        sec = key_combinations / speed_offline
        print(f"Time to crack: {sec/86400/365.25:,.0f} years")
else:
    print("❌ Verification Failed.")

### Task 2: Under the Hood (How TOTP is computed)

In [ ]:
# 1. Absolute Time
current_time = time.time()
print(f"Absolute Time:  {current_time:.2f} (Epoch seconds)")

# 2. Counter Calculation
counter = int(current_time // 30)
print(f"Counter:        {counter} (Steps of 30 seconds)")

# 3. HMAC-SHA1
key_bytes = base64.b32decode(secret, casefold=True)
c_bytes = counter.to_bytes(8, byteorder="big")
mac = hmac.new(key_bytes, c_bytes, "sha1").digest()
print(f"HMAC Result:    {mac.hex()}")

# 4. Dynamic Truncation
offset = mac[-1] & 0x0F
truncated = int.from_bytes(mac[offset : offset + 4], byteorder="big") & 0x7FFFFFFF
otp = truncated % 1_000_000

print(f"\nFinal Code:     {str(otp).zfill(6)}")

---
## 6. Digital Signatures: Public Proof

**Question:** How does the math work inside an RSA signature?

**The Core Math:**
1. Alice has a modulus **$n$**, a public exponent **$e$**, and a private exponent **$d$**.
2. **Signing:** $s = \text{hash}^d \pmod n$ (using Alice's Private Key $d$)
3. **Recovery:** $h = s^e \pmod n$ (using Alice's Public Key $e$)

### Task 1: The Math Inside

In [ ]:
# 1. Generate RSA key
rsa_priv = rsa.generate_private_key(65537, 2048)
rsa_pub = rsa_priv.public_key()

# 2. Extract the numbers
priv_nums = rsa_priv.private_numbers()
pub_nums = rsa_pub.public_numbers()

n = pub_nums.n
e = pub_nums.e
d = priv_nums.d

print(f"Modulus (n):        {str(n)[:30]}...")
print(f"Public Exponent (e): {e}")
print(f"Private Exponent (d):{str(d)[:30]}...")

# 3. Hash a message
message = b"Pay Bob $100"
msg_hash_int = int.from_bytes(hashlib.sha256(message).digest(), "big")
print(f"\nMessage Hash (as int): {str(msg_hash_int)[:30]}...")

# 4. "The Math Inside" - SIGNING (Alice)
# s = m^d mod n
signature_int = pow(msg_hash_int, d, n)
print(f"Signature (s):         {str(signature_int)[:30]}...")

# 5. "The Math Inside" - RECOVERY (The World)
# m = s^e mod n
recovered_hash_int = pow(signature_int, e, n)
print(f"Recovered Hash (m):    {str(recovered_hash_int)[:30]}...")

# 6. Verify Match
if msg_hash_int == recovered_hash_int:
    print("\n✅ MATH MATCHES! (m^d)^e mod n == m")
    print("The original hash was successfully recovered using only the Public Key.")

---
## 7. Real World: Trust on the Web 🌐

How do we trust the Public Key of a website? A **Certificate** is a Public Key signed by a **Trusted Authority (CA)**.

### Task 0: Preparation (Backup Chain)
If we can't connect live, we will use a stored chain.

In [175]:
import ssl, socket

def download_backup():
    print("Attempting to download 'google.crt' backup chain...")
    try:
        context = ssl.create_default_context()
        with socket.create_connection(("google.com", 443), timeout=5) as sock:
            with context.wrap_socket(sock, server_hostname="google.com") as ssock:
                if hasattr(ssock, "get_peer_cert_chain"):
                    bin_certs = ssock.get_peer_cert_chain()
                    with open("google.crt", "wb") as f:
                        for c in bin_certs:
                            f.write(x509.load_der_x509_certificate(c).public_bytes(serialization.Encoding.PEM))
                    print("✅ google.crt successfully created.")
                else:
                    print("⚠️ get_peer_cert_chain not available. Using manual download.")
    except Exception as e:
        print(f"❌ Could not download chain: {e}")

if not os.path.exists("google.crt"):
    download_backup()

### Task 1: The Certificate Chain
Pick a website and see the **Chain of Trust**. Each certificate is signed by the one below it.


In [176]:
hostname = "google.com"

chain = []
try:
    context = ssl.create_default_context()
    with socket.create_connection((hostname, 443), timeout=5) as sock:
        with context.wrap_socket(sock, server_hostname=hostname) as ssock:
            if hasattr(ssock, "get_peer_cert_chain") and ssock.get_peer_cert_chain():
                bin_certs = ssock.get_peer_cert_chain()
                chain = [x509.load_der_x509_certificate(c) for c in bin_certs]
            else: raise Exception("Forcing local cert load for full chain demo.")
except:
    print("Using local backup 'google.crt'...")
    with open("google.crt", "rb") as f:
        certs_data = f.read().split(b"-----END CERTIFICATE-----")
        chain = [x509.load_pem_x509_certificate(c + b"-----END CERTIFICATE-----") for c in certs_data if b"BEGIN CERTIFICATE" in c]

print(f"Chain of Trust for: {hostname}\n" + "="*50)
for i, cert in enumerate(chain):
    indent = "  " * i
    print(f"{indent}CERTIFICATE {i}")
    print(f"{indent}  Subject:    {cert.subject.rfc4514_string()}")
    print(f"{indent}  Issuer:     {cert.issuer.rfc4514_string()}")

    pub = cert.public_key()
    if isinstance(pub, rsa.RSAPublicKey):
        print(f"{indent}  Public Key: RSA {pub.key_size} bits (Exponent: {pub.public_numbers().e})")
    else:
        print(f"{indent}  Public Key: {type(pub).__name__}")

    print(f"{indent}  Signature:  {cert.signature.hex()[:60]}...")

    if i < len(chain) - 1:
        print(f"{indent}   ^--- Signed by Public Key in CERTIFICATE {i+1}")
    else:
        if cert.subject == cert.issuer:
            print(f"{indent}   ^--- This is the ROOT (Self-signed)")
        else:
            print(f"{indent}   ^--- Top of downloaded chain (Root CA is in your OS trust store)")
    print("-" * 50)

Using local backup 'google.crt'...
Chain of Trust for: google.com
CERTIFICATE 0
  Subject:    CN=*.google.com
  Issuer:     CN=WE2,O=Google Trust Services,C=US
  Public Key: ECPublicKey
  Signature:  3045022100e41104b70a3c0d4f7b483ae8e7fc43e936c0f447fc1a3fff7b...
   ^--- Signed by Public Key in CERTIFICATE 1
--------------------------------------------------
  CERTIFICATE 1
    Subject:    CN=WE2,O=Google Trust Services,C=US
    Issuer:     CN=GTS Root R4,O=Google Trust Services LLC,C=US
    Public Key: ECPublicKey
    Signature:  306402300bbdb83655c835a3d2d97d3973d3f7f782b809d1816fe56445db...
     ^--- Signed by Public Key in CERTIFICATE 2
--------------------------------------------------
    CERTIFICATE 2
      Subject:    CN=GTS Root R4,O=Google Trust Services LLC,C=US
      Issuer:     CN=GlobalSign Root CA,OU=Root CA,O=GlobalSign nv-sa,C=BE
      Public Key: ECPublicKey
      Signature:  1842bb0f06d6038796e33f63810f09a4a168480c3922739ef8cb4e2d7f31...
       ^--- Top of downloaded

### Task 2: Verify the Entire Chain
We verify each link in the chain, starting from the website's certificate up to the root.


In [177]:
from cryptography.hazmat.primitives.asymmetric import ec

if len(chain) > 1:
    print(f"Verifying the entire chain for {hostname}...\n")

    for i in range(len(chain) - 1):
        current_cert = chain[i]
        issuer_cert = chain[i+1]

        print(f"Verifying CERTIFICATE {i} (Subject: {current_cert.subject.rfc4514_string()})")
        print(f"  Signed by CERTIFICATE {i+1} (Issuer: {issuer_cert.subject.rfc4514_string()})\n")

        try:
            pub = issuer_cert.public_key()
            if isinstance(pub, rsa.RSAPublicKey):
                pub.verify(
                    current_cert.signature,
                    current_cert.tbs_certificate_bytes,
                    padding.PKCS1v15(),
                    current_cert.signature_hash_algorithm
                )
            elif isinstance(pub, ec.EllipticCurvePublicKey):
                pub.verify(
                    current_cert.signature,
                    current_cert.tbs_certificate_bytes,
                    ec.ECDSA(current_cert.signature_hash_algorithm)
                )
            print(f"✅ LINK {i} VERIFIED: Signature on Cert {i} matches Public Key in Cert {i+1}.")
        except Exception as e:
            print(f"❌ LINK {i} FAILED: {e}")
            print("Chain verification stopped due to a broken link.")
            break
        print("-" * 50)

    root_cert = chain[-1]
    if root_cert.subject == root_cert.issuer:
        print(f"✅ ROOT CERTIFICATE {len(chain)-1} is self-signed (Subject == Issuer).")
    else:
        print(f"ℹ️  CERTIFICATE {len(chain)-1} is cross-signed (not self-signed).")
        print(f"   Issuer: {root_cert.issuer.rfc4514_string()}")
        print("   The ultimate Root CA is trusted via your OS/browser trust store, not sent by the server.")

else:
    print("Error: Chain too short to perform full verification (need at least 2 certs).")

Verifying the entire chain for google.com...

Verifying CERTIFICATE 0 (Subject: CN=*.google.com)
  Signed by CERTIFICATE 1 (Issuer: CN=WE2,O=Google Trust Services,C=US)

✅ LINK 0 VERIFIED: Signature on Cert 0 matches Public Key in Cert 1.
--------------------------------------------------
Verifying CERTIFICATE 1 (Subject: CN=WE2,O=Google Trust Services,C=US)
  Signed by CERTIFICATE 2 (Issuer: CN=GTS Root R4,O=Google Trust Services LLC,C=US)

✅ LINK 1 VERIFIED: Signature on Cert 1 matches Public Key in Cert 2.
--------------------------------------------------
ℹ️  CERTIFICATE 2 is cross-signed (not self-signed).
   Issuer: CN=GlobalSign Root CA,OU=Root CA,O=GlobalSign nv-sa,C=BE
   The ultimate Root CA is trusted via your OS/browser trust store, not sent by the server.


**Final Takeaway:**
- **Passwords:** Static and weak.
- **TOTP:** Dynamic but still shared.
- **Signatures:** Proof without sharing secrets.
- **Certificates:** Trusted chains of signatures that link a website to a Root Authority.
